# Лабораторная работа № 4. Векторизация и broadcasting: финансовое моделирование

**Кейс:** как изменить скидочную политику и выполнить план продаж.
**Данные:** `sales` — продажи 4 регионов (строки) по 6 неделям (столбцы), `target` — недельный план, `cost_rate` — доля себестоимости в выручке региона.
**Ограничения:** в обязательных заданиях нет циклов `for`/`while`, Pandas и ручного перебора; только векторизованные операции NumPy, маски и агрегации.

In [1]:
import numpy as np
np.set_printoptions(suppress=True, precision=2, linewidth=140)

sales = np.array([
    [12000, 15000, 18000, 21000, 19000, 23000],
    [9000, 11000, 14000, 16000, 13000, 15000],
    [20000, 22000, 19000, 25000, 24000, 26000],
    [8000, 13000, 17000, 15000, 14000, 18000]
])
target = np.array([15000, 16000, 17000, 18000, 20000, 22000])
cost_rate = np.array([0.70, 0.75, 0.65, 0.80])
regions = np.array(["Регион 1", "Регион 2", "Регион 3", "Регион 4"])
print(sales.shape, target.shape, cost_rate.shape)

(4, 6) (6,) (4,)


Формы: `sales` — `(4, 6)`, `target` — `(6,)`, `cost_rate` — `(4,)`. Деньги условные (ден. ед.).

## Задание 1. Выполнение плана

`sales - target` — это broadcasting: массив `target` формы `(6,)` растягивается по строкам, поэтому каждая неделя сравнивается с планом своей недели. Процент выполнения = `sales / target * 100`.

In [2]:
deviation = sales - target
met = sales >= target
completion = sales / target * 100
print("Отклонение от плана:\n", deviation)
print("План выполнен:\n", met)
print("Выполнение плана, %:\n", completion)
print("Недель с выполненным планом по регионам:", met.sum(axis=1))

Отклонение от плана:
 [[-3000 -1000  1000  3000 -1000  1000]
 [-6000 -5000 -3000 -2000 -7000 -7000]
 [ 5000  6000  2000  7000  4000  4000]
 [-7000 -3000     0 -3000 -6000 -4000]]
План выполнен:
 [[False False  True  True False  True]
 [False False False False False False]
 [ True  True  True  True  True  True]
 [False False  True False False False]]
Выполнение плана, %:
 [[ 80.    93.75 105.88 116.67  95.   104.55]
 [ 60.    68.75  82.35  88.89  65.    68.18]
 [133.33 137.5  111.76 138.89 120.   118.18]
 [ 53.33  81.25 100.    83.33  70.    81.82]]
Недель с выполненным планом по регионам: [3 0 6 1]


**Вывод.** План выполнен в 3 из 6 недель у Региона 1, в 0 из 6 у Региона 2, во всех 6 у Региона 3 и лишь в 1 неделе (3-й, ровно 100%) у Региона 4. Регион 3 стабильно перевыполняет план (112–139%), Регион 2 не дотягивает ни разу (60–89%).

## Задание 2. Оценка регионов

Суммы и средние считаются по оси `axis=1` (по неделям). `keepdims=True` сохраняет форму `(4, 1)`, поэтому `sales - mean` корректно вычитает среднее каждого региона из всех его недель. Для сравнения стабильности абсолютного стандартного отклонения мало — регионы разного масштаба, поэтому использован коэффициент вариации (std / среднее).

In [3]:
total = sales.sum(axis=1)
mean = sales.mean(axis=1, keepdims=True)
dev_mean = sales - mean
std = sales.std(axis=1)
cv = std / mean.ravel() * 100
print("Выручка по регионам:", total)
print("Среднее в неделю:", mean.ravel())
print("Отклонение от среднего:\n", dev_mean)
print("Стд. отклонение:", std)
print("Коэф. вариации, %:", cv)
print("Самый стабильный:", regions[cv.argmin()], "| наибольшие колебания:", regions[cv.argmax()])

Выручка по регионам: [108000  78000 136000  85000]
Среднее в неделю: [18000.   13000.   22666.67 14166.67]
Отклонение от среднего:
 [[-6000.   -3000.       0.    3000.    1000.    5000.  ]
 [-4000.   -2000.    1000.    3000.       0.    2000.  ]
 [-2666.67  -666.67 -3666.67  2333.33  1333.33  3333.33]
 [-6166.67 -1166.67  2833.33   833.33  -166.67  3833.33]]
Стд. отклонение: [3651.48 2380.48 2560.38 3236.08]
Коэф. вариации, %: [20.29 18.31 11.3  22.84]
Самый стабильный: Регион 3 | наибольшие колебания: Регион 4


**Вывод.** Выручка: Регион 3 — 136 000, Регион 1 — 108 000, Регион 4 — 85 000, Регион 2 — 78 000. **Наиболее стабилен Регион 3** (коэффициент вариации 11,3%), **наибольшие колебания у Региона 4** (22,8%). По абсолютному std самый большой разброс у Региона 1 (3 651), но относительно его среднего уровня он меньше, чем у Региона 4.

## Задание 3. Финансовая модель

`cost_rate` имеет форму `(4,)`, а `sales` — `(4, 6)`. Напрямую они несовместимы (сравниваются хвостовые оси 6 и 4), поэтому `cost_rate[:, np.newaxis]` превращает его в `(4, 1)`. Тогда ось длины 1 растягивается до 6: ставка региона применяется ко всем его неделям. Себестоимость = `sales * cost_rate`, валовая прибыль = `sales - cost`.

In [4]:
cost = sales * cost_rate[:, np.newaxis]
gross = sales - cost
print(cost_rate.shape, cost_rate[:, np.newaxis].shape, sales.shape, cost.shape)
print("Себестоимость:\n", cost)
print("Валовая прибыль:\n", gross)
print("Прибыль по регионам:", gross.sum(axis=1), "| маржа, %:", gross.sum(axis=1) / total * 100)

(4,) (4, 1) (4, 6) (4, 6)
Себестоимость:
 [[ 8400. 10500. 12600. 14700. 13300. 16100.]
 [ 6750.  8250. 10500. 12000.  9750. 11250.]
 [13000. 14300. 12350. 16250. 15600. 16900.]
 [ 6400. 10400. 13600. 12000. 11200. 14400.]]
Валовая прибыль:
 [[3600. 4500. 5400. 6300. 5700. 6900.]
 [2250. 2750. 3500. 4000. 3250. 3750.]
 [7000. 7700. 6650. 8750. 8400. 9100.]
 [1600. 2600. 3400. 3000. 2800. 3600.]]
Прибыль по регионам: [32400. 19500. 47600. 17000.] | маржа, %: [30. 25. 35. 20.]


**Вывод.** Валовая прибыль: Регион 3 — 47 600 (маржа 35%), Регион 1 — 32 400 (30%), Регион 2 — 19 500 (25%), Регион 4 — 17 000 (20%). Всего 116 500. Самая высокая выручка не всегда даёт самую высокую маржу: у Региона 4 выручка больше, чем у Региона 2, но прибыль меньше из-за высокой себестоимости.

## Задание 4. Сценарии скидок

Три сценария скидки `[0, 5%, 10%]` строятся одним выражением с broadcasting: массив `(3, 1, 1)` умножается на `(1, 4, 6)` и даёт `(3, 4, 6)` — по одной матрице 4×6 на сценарий. Объём продаж по условию не меняется, поэтому себестоимость остаётся прежней, а скидка уменьшает только выручку.

In [5]:
discount = np.array([0.0, 0.05, 0.10])
revenue_s = sales[np.newaxis, :, :] * (1 - discount)[:, np.newaxis, np.newaxis]
gross_s = revenue_s - cost[np.newaxis, :, :]
print(revenue_s.shape, gross_s.shape)
print("Выручка по сценариям:", revenue_s.sum(axis=(1, 2)))
print("Прибыль по сценариям:", gross_s.sum(axis=(1, 2)))
print("Прибыль по регионам и сценариям:\n", gross_s.sum(axis=2))
print("Изменение прибыли к базе, %:", (gross_s.sum(axis=(1, 2)) / gross_s[0].sum() - 1) * 100)

(3, 4, 6) (3, 4, 6)
Выручка по сценариям: [407000. 386650. 366300.]
Прибыль по сценариям: [116500.  96150.  75800.]
Прибыль по регионам и сценариям:
 [[32400. 19500. 47600. 17000.]
 [27000. 15600. 40800. 12750.]
 [21600. 11700. 34000.  8500.]]
Изменение прибыли к базе, %: [  0.   -17.47 -34.94]


**Вывод.** Выручка: 407 000 → 386 650 → 366 300. Валовая прибыль: 116 500 → 96 150 (−17,5%) → 75 800 (−34,9%). Скидка 5% стоит 20 350 — ровно 5% выручки, и вся она вычитается из прибыли, потому что себестоимость не меняется.

**Почему расчёт ненадёжен.** Мы предположили, что объём продаж не меняется. На практике скидка влияет на спрос (может увеличить его или лишь отдать скидку тем, кто купил бы и так), а себестоимость единицы и общая себестоимость зависят от объёма. Нет данных об эластичности спроса, реакции конкурентов, сезонности и повторных покупках, поэтому сценарий показывает только цену скидки при неизменном объёме, а не фактический результат кампании.

## Задание 5. Поиск финансового риска

Минимум по региону — `min(axis=1)` и `argmin`. Чувствительность к скидке — относительное падение прибыли при скидке 10% по сравнению с базой.

In [6]:
print("Минимальная прибыль по региону:", gross.min(axis=1), "в неделях", gross.argmin(axis=1) + 1)
print("Минимум в сценарии -10%:", gross_s[2].min(), "| отрицательных значений:", (gross_s < 0).sum())
loss = (1 - gross_s[2].sum(axis=1) / gross_s[0].sum(axis=1)) * 100
print("Падение прибыли при скидке 10%, % по регионам:", loss)
print("Самый чувствительный:", regions[loss.argmax()])

Минимальная прибыль по региону: [3600. 2250. 6650. 1600.] в неделях [1 1 3 1]
Минимум в сценарии -10%: 800.0 | отрицательных значений: 0
Падение прибыли при скидке 10%, % по регионам: [33.33 40.   28.57 50.  ]
Самый чувствительный: Регион 4


**Вывод.** Минимальная прибыль в базовом сценарии: Регион 4 — 1 600 (неделя 1), Регион 2 — 2 250 (неделя 1), Регион 1 — 3 600 (неделя 1), Регион 3 — 6 650 (неделя 3). Отрицательной прибыли нет даже при скидке 10% (минимум 800 — Регион 4, неделя 1), но запас прочности там минимальный. **Наиболее чувствителен к скидкам Регион 4**: при скидке 10% его прибыль падает на 50%, у Региона 2 — на 40%, у Региона 1 — на 33%, у Региона 3 — на 29%.

## Задание 6. Анализ динамики

`np.diff(sales, axis=1)` даёт абсолютное изменение между соседними неделями. Процентное изменение = изменение / предыдущее значение. Деление защищено: `np.divide(..., where=prev != 0)` не делит там, где предыдущее значение равно 0, и оставляет `NaN` вместо `inf` и предупреждения. В этих данных нулей нет, но защита нужна для корректности.

In [7]:
change = np.diff(sales, axis=1)
prev = sales[:, :-1]
pct = np.divide(change, prev, out=np.full(change.shape, np.nan), where=prev != 0) * 100
print("Изменение:\n", change)
print("Изменение, %:\n", pct)
print("Недель роста:", (change > 0).sum(axis=1), "| недель снижения:", (change < 0).sum(axis=1))
print("Худшая неделя (по регионам, номер перехода):", change.argmin(axis=1) + 1, change.min(axis=1))

Изменение:
 [[ 3000  3000  3000 -2000  4000]
 [ 2000  3000  2000 -3000  2000]
 [ 2000 -3000  6000 -1000  2000]
 [ 5000  4000 -2000 -1000  4000]]
Изменение, %:
 [[ 25.    20.    16.67  -9.52  21.05]
 [ 22.22  27.27  14.29 -18.75  15.38]
 [ 10.   -13.64  31.58  -4.     8.33]
 [ 62.5   30.77 -11.76  -6.67  28.57]]
Недель роста: [4 4 3 3] | недель снижения: [1 1 2 2]
Худшая неделя (по регионам, номер перехода): [4 4 2 3] [-2000 -3000 -3000 -2000]


**Вывод.** Рост преобладает: 4 недели роста и 1 снижение у Регионов 1 и 2, по 3 роста и 2 снижения у Регионов 3 и 4. **Во всех регионах продажи падают при переходе с 4-й на 5-ю неделю** (от −1 000 до −3 000), а у Региона 2 это самое сильное падение (−18,75%). Самый резкий рост — Регион 4 в начале (+62,5%), но с низкой базы.

## Задание 7. Сравнение с планом

Два показателя: число недель ниже плана (`(sales < target).sum(axis=1)`) и суммарное отклонение (`deviation.sum(axis=1)`). Дополнительно — сумма только отрицательных отклонений (недобор).

In [8]:
below = sales < target
print("Недель ниже плана:", below.sum(axis=1))
print("Суммарное отклонение:", deviation.sum(axis=1))
print("Суммарный недобор (только отрицательные):", np.where(deviation < 0, deviation, 0).sum(axis=1))

Недель ниже плана: [3 6 0 5]
Суммарное отклонение: [     0 -30000  28000 -23000]
Суммарный недобор (только отрицательные): [ -5000 -30000      0 -23000]


**Вывод.** Чаще всего план не выполняет **Регион 2** (6 из 6 недель), затем Регион 4 (5 из 6) и Регион 1 (3 из 6); Регион 3 — ни разу. По суммарному отклонению хуже всего тоже Регион 2 (−30 000), затем Регион 4 (−23 000).

**Почему показатели различаются.** Число недель показывает частоту невыполнения, но не размер. Суммарное отклонение учитывает размер, но перевыполнение одних недель гасит недобор других: у Региона 1 три недели ниже плана, а суммарное отклонение 0, потому что недобор (−5 000) покрыт перевыполнением (+5 000). Поэтому полезно смотреть оба показателя вместе, а также отдельно сумму недобора.

## Задание 8. Рекомендации по скидкам

Добавлен показатель: на сколько процентов должен вырасти объём продаж, чтобы скидка не уменьшила валовую прибыль региона. При марже `m` и скидке `d` прибыль с единицы падает с `m` до `m − d`, поэтому нужный рост объёма = `m / (m − d) − 1`.

In [9]:
margin = 1 - cost_rate
base = gross_s[0].sum()
print("Потеря прибыли при скидке 5%:", base - gross_s[1].sum(), "| при 10%:", base - gross_s[2].sum())
print("Маржа по регионам, %:", margin * 100)
need5 = margin / (margin - 0.05) - 1
need10 = margin / (margin - 0.10) - 1
print("Нужный рост объёма при скидке 5%, % :", need5 * 100)
print("Нужный рост объёма при скидке 10%, %:", need10 * 100)

Потеря прибыли при скидке 5%: 20350.0 | при 10%: 40700.0
Маржа по регионам, %: [30. 25. 35. 20.]
Нужный рост объёма при скидке 5%, % : [20.   25.   16.67 33.33]
Нужный рост объёма при скидке 10%, %: [ 50.    66.67  40.   100.  ]


**Рекомендация.** Общая скидка на всех — плохая идея: она снижает прибыль на 17,5% (5%) и на 34,9% (10%) и окупится только при заметном росте спроса. Нужный рост объёма при скидке 5%: Регион 3 +16,7%, Регион 1 +20%, Регион 2 +25%, Регион 4 +33,3%; при скидке 10% — от +40% (Регион 3) до +100% (Регион 4).

- **Регион 4** (маржа 20%, самый чувствительный) — не давать скидки, сначала снижать себестоимость.
- **Регион 3** (маржа 35%) — самый безопасный для пилота 5% скидки, но он и так перевыполняет план, поэтому скидка там нужна меньше всего.
- **Регион 2** не выполняет план ни одной недели, но у него маржа 25%, поэтому пилот 5% возможен только при ожидаемом росте объёма больше 25%.
- Лучше всего начать с небольшого пилота в одном регионе и сравнить результат с контрольным.

**Каких данных не хватает:** эластичность спроса (как объём реагирует на цену), объёмы продаж при прошлых скидках, постоянные расходы, реакция конкурентов, структура товаров (у разных товаров разная маржа), сезонность, доля повторных покупателей.

## Задание 9. Диагностика broadcasting

Правило: формы сравниваются справа налево; оси совместимы, если они равны или одна из них равна 1; недостающие слева оси считаются равными 1.

In [10]:
m = np.ones((4, 6))

def check(shape):
    try:
        print(shape, "-> OK, результат", (m + np.ones(shape)).shape)
    except ValueError as e:
        print(shape, "-> ОШИБКА:", e)

check((6,))
check((4,))
check((4, 1))
check((4, 6))
fixed = m + np.arange(4).reshape(4, 1)
print("Исправление reshape(4, 1):", fixed.shape)

(6,) -> OK, результат (4, 6)
(4,) -> ОШИБКА: operands could not be broadcast together with shapes (4,6) (4,) 
(4, 1) -> OK, результат (4, 6)
(4, 6) -> OK, результат (4, 6)
Исправление reshape(4, 1): (4, 6)


| Случай | Результат | Объяснение |
|---|---|---|
| `(4, 6)` и `(6,)` | корректно → `(4, 6)` | `(6,)` считается как `(1, 6)` и растягивается по строкам: один массив применяется к каждому региону (как `target` по неделям). |
| `(4, 6)` и `(4,)` | **ошибка** | справа сравниваются 6 и 4: не равны и ни одно не равно 1. Исправление, если массив — значения по регионам (как `cost_rate`): `reshape(4, 1)` или `[:, np.newaxis]`. |
| `(4, 6)` и `(4, 1)` | корректно → `(4, 6)` | ось длины 1 растягивается до 6: значение региона применяется ко всем его неделям. |
| `(4, 6)` и `(4, 6)` | корректно → `(4, 6)` | формы одинаковы: обычное поэлементное действие, broadcasting не нужен. |

Осторожно: с квадратной матрицей `(4, 4)` массив `(4,)` не вызовет ошибки, но будет применён по строкам, а не по регионам, то есть результат молча окажется неверным. Поэтому формы нужно проверять явно.

## Дополнительное задание. Сценарная модель себестоимости и скидки

Сетка 7×7: уровни себестоимости (65–95%) по строкам и скидки (0–30%) по столбцам. Для каждой пары валовая прибыль на всей выручке (407 000) = `выручка × ((1 − скидка) − себестоимость)`. Обе оси задаются массивами `(7, 1)` и `(1, 7)`, и broadcasting строит всю таблицу без циклов. Значения округлены до 4 знаков, чтобы ошибки плавающей точки не меняли знак на границе.

In [11]:
total_sales = sales.sum()
cost_levels = np.array([0.65, 0.70, 0.75, 0.80, 0.85, 0.90, 0.95])
disc_levels = np.array([0.0, 0.05, 0.10, 0.15, 0.20, 0.25, 0.30])
unit_margin = np.round((1 - disc_levels)[np.newaxis, :] - cost_levels[:, np.newaxis], 4)
grid = total_sales * unit_margin
print(grid)
print("Отрицательных комбинаций:", (grid < 0).sum(), "из", grid.size)
neg_c, neg_d = np.where(grid < 0)
print(np.c_[cost_levels[neg_c], disc_levels[neg_d]])

[[ 142450.  122100.  101750.   81400.   61050.   40700.   20350.]
 [ 122100.  101750.   81400.   61050.   40700.   20350.       0.]
 [ 101750.   81400.   61050.   40700.   20350.       0.  -20350.]
 [  81400.   61050.   40700.   20350.       0.  -20350.  -40700.]
 [  61050.   40700.   20350.       0.  -20350.  -40700.  -61050.]
 [  40700.   20350.       0.  -20350.  -40700.  -61050.  -81400.]
 [  20350.       0.  -20350.  -40700.  -61050.  -81400. -101750.]]
Отрицательных комбинаций: 15 из 49
[[0.75 0.3 ]
 [0.8  0.25]
 [0.8  0.3 ]
 [0.85 0.2 ]
 [0.85 0.25]
 [0.85 0.3 ]
 [0.9  0.15]
 [0.9  0.2 ]
 [0.9  0.25]
 [0.9  0.3 ]
 [0.95 0.1 ]
 [0.95 0.15]
 [0.95 0.2 ]
 [0.95 0.25]
 [0.95 0.3 ]]


**Вывод.** Прибыль отрицательна в 15 из 49 комбинаций. Условие убытка простое: скидка больше маржи, то есть `скидка > 1 − себестоимость`; на границе (прибыль 0) например 70% / 30% или 80% / 20%.

**Самые опасные комбинации:** высокая себестоимость (≥ 90%) при любой скидке от 10–15% и скидки 25–30% при себестоимости от 80%. Для текущих регионов (себестоимость 65–80%) убыток начинается при скидке выше 20% у Региона 4 (себестоимость 80%), выше 25% у Региона 2 (75%), выше 30% у Региона 1 (70%) и выше 35% у Региона 3 (65%). Значит скидки до 10%, которые обсуждались, убыточными не станут, но съедят от трети до половины прибыли у Регионов 1, 2 и 4.

*Упрощение: сетка использует одну себестоимость для всей выручки, а не по регионам.*